# CNN Architectures Deep Dive

Understanding the evolution and design of Convolutional Neural Network architectures.

## Learning Objectives

- Understand key innovations in CNN architecture design
- Implement and compare VGG, ResNet, and EfficientNet
- Analyze model complexity (parameters, FLOPs)
- Visualize feature hierarchies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

import torchvision
from torchvision import models
from torchvision.transforms import v2
from torchvision import datasets

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. CNN Architecture Evolution

Key milestones in CNN development:

| Year | Architecture | Key Innovation | ImageNet Top-5 |
|------|-------------|----------------|-----------------|
| 2012 | AlexNet | ReLU, Dropout, GPU training | 16.4% |
| 2014 | VGG | Deeper with 3x3 filters | 7.3% |
| 2015 | ResNet | Skip connections | 3.6% |
| 2017 | DenseNet | Dense connections | 3.5% |
| 2019 | EfficientNet | Compound scaling | 2.9% |
| 2020 | ViT | Transformers for vision | 1.8% |

## 2. VGG: Deep but Simple

VGG demonstrated that **depth matters**. Key design principles:
- Stack small 3x3 filters (two 3x3 = one 5x5 receptive field)
- Double channels after each pooling
- Simple, uniform architecture

In [ ]:
# Simplified VGG-like block
class VGGBlock(nn.Module):
    """VGG-style block: Conv -> BN -> ReLU repeated n times, then MaxPool."""
    
    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        layers = []
        for i in range(num_convs):
            layers.append(nn.Conv2d(
                in_channels if i == 0 else out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)


class MiniVGG(nn.Module):
    """Mini VGG for CIFAR-10 (32x32 images)."""
    
    def __init__(self, num_classes=10):
        super().__init__()
        # Config: (out_channels, num_convs)
        config = [(64, 2), (128, 2), (256, 3), (512, 3)]
        
        blocks = []
        in_channels = 3
        for out_channels, num_convs in config:
            blocks.append(VGGBlock(in_channels, out_channels, num_convs))
            in_channels = out_channels
        
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Create and analyze
vgg_model = MiniVGG()
print("Mini VGG Architecture:")
print(vgg_model)

# Test forward pass
x = torch.randn(1, 3, 32, 32)
y = vgg_model(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {y.shape}")

In [ ]:
# Count parameters
def count_parameters(model):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_parameters_by_layer(model):
    """Count parameters per layer type."""
    counts = {}
    for name, module in model.named_modules():
        layer_type = type(module).__name__
        params = sum(p.numel() for p in module.parameters(recurse=False))
        if params > 0:
            counts[layer_type] = counts.get(layer_type, 0) + params
    return counts

print(f"Total parameters: {count_parameters(vgg_model):,}")
print("\nParameters by layer type:")
for layer_type, count in count_parameters_by_layer(vgg_model).items():
    print(f"  {layer_type}: {count:,}")

## 3. ResNet: Skip Connections

ResNet solved the **degradation problem** with skip (residual) connections:

```
y = F(x) + x  # Residual connection
```

This allows training of very deep networks (50, 101, 152+ layers).

In [ ]:
class BasicBlock(nn.Module):
    """Basic ResNet block with skip connection."""
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection with optional downsample
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out += identity  # Skip connection!
        out = F.relu(out)
        
        return out


class Bottleneck(nn.Module):
    """Bottleneck block for deeper ResNets (50, 101, 152)."""
    expansion = 4
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # 1x1 -> 3x3 -> 1x1 (bottleneck design)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        
        out += identity
        out = F.relu(out)
        
        return out

In [ ]:
class MiniResNet(nn.Module):
    """Mini ResNet for CIFAR-10."""
    
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64
        
        # Initial conv (no stride for 32x32 images)
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Residual layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


# Create ResNet-18 equivalent
resnet_model = MiniResNet(BasicBlock, [2, 2, 2, 2])
print(f"Mini ResNet-18 parameters: {count_parameters(resnet_model):,}")

# Test forward pass
x = torch.randn(1, 3, 32, 32)
y = resnet_model(x)
print(f"Input: {x.shape} -> Output: {y.shape}")

In [ ]:
# Visualize skip connection effect
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Without skip connection (gradient vanishing)
depth = np.arange(1, 50)
gradient_no_skip = 0.9 ** depth  # Gradients diminish

# With skip connection (gradient preserved)
gradient_with_skip = 0.9 ** depth + 1.0  # Identity preserves gradient
gradient_with_skip = gradient_with_skip / gradient_with_skip.max()

axes[0].plot(depth, gradient_no_skip, 'r-', linewidth=2, label='Without Skip')
axes[0].plot(depth, gradient_with_skip, 'g-', linewidth=2, label='With Skip')
axes[0].set_xlabel('Layer Depth')
axes[0].set_ylabel('Relative Gradient Magnitude')
axes[0].set_title('Gradient Flow Through Network')
axes[0].legend()
axes[0].set_yscale('log')

# Architecture diagram
axes[1].text(0.5, 0.9, 'Standard Block', ha='center', fontsize=12, weight='bold')
axes[1].text(0.5, 0.75, 'x → Conv → BN → ReLU → Conv → BN → ReLU → y', ha='center', fontsize=10)
axes[1].text(0.5, 0.5, 'Residual Block', ha='center', fontsize=12, weight='bold')
axes[1].text(0.5, 0.35, 'x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU → y', ha='center', fontsize=10)
axes[1].text(0.5, 0.15, 'y = F(x) + x  (Identity Shortcut)', ha='center', fontsize=11, style='italic')
axes[1].axis('off')
axes[1].set_title('Block Comparison')

plt.tight_layout()
plt.show()

## 4. EfficientNet: Compound Scaling

EfficientNet introduced **compound scaling**: scale depth, width, and resolution together.

Uses **Mobile Inverted Bottleneck (MBConv)** blocks with:
- Depthwise separable convolutions
- Squeeze-and-Excitation attention
- Inverted residuals

In [ ]:
class SqueezeExcitation(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    
    def __init__(self, channels, reduction=4):
        super().__init__()
        reduced = channels // reduction
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(channels, reduced)
        self.fc2 = nn.Linear(reduced, channels)
    
    def forward(self, x):
        b, c, _, _ = x.shape
        # Squeeze
        y = self.pool(x).view(b, c)
        # Excitation
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y))
        # Scale
        return x * y.view(b, c, 1, 1)


class MBConv(nn.Module):
    """Mobile Inverted Bottleneck Conv block."""
    
    def __init__(self, in_channels, out_channels, expand_ratio, stride, kernel_size=3):
        super().__init__()
        self.stride = stride
        self.use_residual = stride == 1 and in_channels == out_channels
        
        hidden = in_channels * expand_ratio
        
        layers = []
        
        # Expansion
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden, 1, bias=False),
                nn.BatchNorm2d(hidden),
                nn.SiLU(inplace=True)  # Swish activation
            ])
        
        # Depthwise conv
        layers.extend([
            nn.Conv2d(hidden, hidden, kernel_size, stride, kernel_size//2, 
                     groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.SiLU(inplace=True)
        ])
        
        # Squeeze-and-Excitation
        layers.append(SqueezeExcitation(hidden))
        
        # Projection
        layers.extend([
            nn.Conv2d(hidden, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        out = self.block(x)
        if self.use_residual:
            out = out + x
        return out


# Demonstrate SE block
se = SqueezeExcitation(64)
x = torch.randn(1, 64, 8, 8)
y = se(x)
print(f"SE Block: {x.shape} -> {y.shape}")

# Demonstrate MBConv
mbconv = MBConv(32, 64, expand_ratio=6, stride=2)
x = torch.randn(1, 32, 16, 16)
y = mbconv(x)
print(f"MBConv: {x.shape} -> {y.shape}")

In [ ]:
# Compare depthwise separable vs standard convolution
in_channels, out_channels = 64, 128
kernel_size = 3
input_size = 32

# Standard convolution
standard_conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=1)
standard_params = sum(p.numel() for p in standard_conv.parameters())

# Depthwise separable convolution
depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, padding=1, groups=in_channels)
pointwise = nn.Conv2d(in_channels, out_channels, 1)
separable_params = sum(p.numel() for p in depthwise.parameters()) + \
                   sum(p.numel() for p in pointwise.parameters())

print("=== Depthwise Separable vs Standard Convolution ===")
print(f"\nInput: {in_channels} channels, Output: {out_channels} channels, Kernel: {kernel_size}x{kernel_size}")
print(f"\nStandard Conv params:    {standard_params:,}")
print(f"Depthwise Separable params: {separable_params:,}")
print(f"\nReduction factor: {standard_params / separable_params:.2f}x fewer parameters")

## 5. Using Pre-trained Models

torchvision provides pre-trained models with modern weight loading API.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import vgg16, VGG16_Weights

# Load pre-trained models
print("Loading pre-trained models...")

# Modern API: use Weights enum
resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
efficientnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

# Compare model sizes
models_info = {
    'VGG-16': vgg,
    'ResNet-18': resnet,
    'EfficientNet-B0': efficientnet
}

print("\n=== Model Comparison ===")
for name, model in models_info.items():
    params = count_parameters(model)
    print(f"{name}: {params:,} parameters ({params/1e6:.1f}M)")

In [ ]:
# Analyze layer distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (name, model) in zip(axes, models_info.items()):
    layer_params = count_parameters_by_layer(model)
    
    # Filter significant layers
    total = sum(layer_params.values())
    filtered = {k: v for k, v in layer_params.items() if v > total * 0.01}
    
    labels = list(filtered.keys())
    sizes = list(filtered.values())
    
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(f'{name}\n({sum(sizes)/1e6:.1f}M params)')

plt.suptitle('Parameter Distribution by Layer Type', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature Visualization

Visualize what different layers learn by examining activations.

In [ ]:
# Get a sample image from CIFAR-10
cifar = datasets.CIFAR10(root='./datasets', train=False, download=True)
sample_img, label = cifar[3000]  # Get a dog image
sample_tensor = v2.functional.to_image(sample_img)

# Prepare for model (resize to 224x224)
transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])
input_tensor = transform(sample_tensor).unsqueeze(0)

print(f"Sample class: {cifar.classes[label]}")
plt.figure(figsize=(4, 4))
plt.imshow(sample_img)
plt.title(f'Input Image: {cifar.classes[label]}')
plt.axis('off')
plt.show()

In [ ]:
# Hook to capture intermediate activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks on ResNet layers
resnet.eval()
hooks = []
hooks.append(resnet.layer1.register_forward_hook(get_activation('layer1')))
hooks.append(resnet.layer2.register_forward_hook(get_activation('layer2')))
hooks.append(resnet.layer3.register_forward_hook(get_activation('layer3')))
hooks.append(resnet.layer4.register_forward_hook(get_activation('layer4')))

# Forward pass
with torch.no_grad():
    output = resnet(input_tensor)

# Remove hooks
for hook in hooks:
    hook.remove()

# Visualize activations
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, act) in enumerate(activations.items()):
    # Show first few channels
    for j in range(2):
        ax = axes[j, i]
        channel = act[0, j].cpu().numpy()
        ax.imshow(channel, cmap='viridis')
        ax.set_title(f'{name} ch{j}\n{act.shape[2]}x{act.shape[3]}', fontsize=10)
        ax.axis('off')

plt.suptitle('ResNet-18 Feature Maps at Different Depths', fontsize=14)
plt.tight_layout()
plt.show()

# Print activation shapes
print("\nActivation shapes through the network:")
for name, act in activations.items():
    print(f"  {name}: {tuple(act.shape)}")

## 7. Model Inference

Use pre-trained models for inference with proper preprocessing.

In [ ]:
# Get preprocessing from weights
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

# Prepare input
input_batch = preprocess(sample_tensor).unsqueeze(0)

# Inference
resnet.eval()
with torch.no_grad():
    output = resnet(input_batch)

# Get predictions
probabilities = torch.softmax(output, dim=1)
top5_prob, top5_idx = probabilities.topk(5)

# Get class names from weights
categories = weights.meta['categories']

print("Top 5 Predictions:")
for i in range(5):
    idx = top5_idx[0, i].item()
    prob = top5_prob[0, i].item()
    print(f"  {i+1}. {categories[idx]}: {prob:.1%}")

## 8. Architecture Selection Guide

Choose based on your constraints:

In [ ]:
# Architecture selection summary
selection_guide = """
=== CNN Architecture Selection Guide ===

┌────────────────────┬─────────────────────────────────────────┐
│ Scenario           │ Recommended Architecture                │
├────────────────────┼─────────────────────────────────────────┤
│ Mobile/Edge        │ MobileNetV3, EfficientNet-B0           │
│ Good accuracy      │ ResNet-50, EfficientNet-B4             │
│ Best accuracy      │ EfficientNet-B7, ViT-L, ConvNeXt       │
│ Limited memory     │ ShuffleNetV2, MobileNetV2              │
│ Fast training      │ ResNet-18, ResNet-34                   │
│ Transfer learning  │ ResNet-50, EfficientNet-B4             │
└────────────────────┴─────────────────────────────────────────┘

Key Metrics to Consider:
1. Parameters (memory footprint)
2. FLOPs (computational cost)
3. Latency (inference time)
4. Accuracy (ImageNet top-1/top-5)
"""
print(selection_guide)

## Summary

### Key Architectural Innovations

| Architecture | Key Innovation | Main Benefit |
|-------------|----------------|-------------|
| VGG | Deep stacked 3x3 convs | Showed depth matters |
| ResNet | Skip connections | Enables very deep networks |
| EfficientNet | Compound scaling + MBConv | Best accuracy/efficiency |

### Implementation Tips

1. **Start with pre-trained models** - Transfer learning is almost always better
2. **Use batch normalization** - Stabilizes training
3. **Skip connections help** - Add them in deep networks
4. **Depthwise separable convs** - Great for efficiency
5. **SE blocks** - Cheap attention mechanism

In [ ]:
print("Notebook completed successfully!")